In [15]:
import os
import re
import pandas as pd
import numpy as np

try:
    import pyarrow
    _has_parquet = True
except ImportError:
    _has_parquet = False

# NASDAQ-100 Membership Tracking (2001-2026)

This notebook:
1. Parses NDX ADD/DEL events from `ndx_changes` file
2. Builds daily membership matrices (full and yfinance-only)
3. Combines individual ticker CSVs into `close_prices` matrix
4. Creates two tradable masks:
   - `tradable_mask_full.csv`: All tickers (for plotting constituent count)
   - `tradable_mask.csv`: Only yfinance-available tickers (for RL)

In [16]:
# Configuration
DATA_START_DATE = pd.Timestamp("2003-01-01")  # Start from yfinance data availability
DATA_END_DATE = pd.Timestamp("2026-03-15")    # End date for data

print(f"Building membership matrices from {DATA_START_DATE.date()} to {DATA_END_DATE.date()}")

Building membership matrices from 2003-01-01 to 2026-03-15


In [17]:
# Project paths
BASE = os.path.abspath(os.getcwd())
if os.path.basename(BASE) == "Data":
    BASE = os.path.dirname(BASE)

RAW_DIR = os.path.join(BASE, "Data", "Outputs", "Raw_Data")
RL_DIR = os.path.join(BASE, "Data", "Outputs", "RL_Needs")

for d in [RAW_DIR, RL_DIR]:
    os.makedirs(d, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("RL_DIR:", RL_DIR)

RAW_DIR: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/Raw_Data
RL_DIR: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/RL_Needs


## STEP 1: Parse ADD/DEL Events from ndx_changes

In [18]:
def normalize_ticker(raw: str) -> str:
    """Take first token before space (ignore UW/UQ/UN/UL/Equity)."""
    s = (raw or "").strip()
    if not s:
        return ""
    return s.split()[0]


def extract_tickers_from_line(line: str) -> list:
    """Extract ticker symbols from a line. Handles +, -, *, • and multiple tickers."""
    tickers = []
    # Split by common separators (+, -, *, bullet)
    parts = re.split(r"[+*•\-]", line)
    for part in parts:
        t = normalize_ticker(part)
        if t and t not in ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT"):
            if not re.match(r"^\d+$", t):  # skip pure numbers (e.g. 101, 102)
                tickers.append(t)
    # If no separator, treat whole line as one ticker block
    if not tickers and line.strip():
        t = normalize_ticker(line)
        if t and t not in ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT"):
            if not re.match(r"^\d+$", t):
                tickers.append(t)
    return tickers


def parse_ndx_log(filepath: str) -> pd.DataFrame:
    """Parse NDX ADD/DEL log into events DataFrame."""
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"NDX log not found: {filepath}")
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    events = []
    current_date = None
    in_add = False
    in_del = False
    date_re = re.compile(r"(?:PERIOD\s*:\s*)?(\d{4}-\d{2}-\d{2})")

    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Detect PERIOD / date-only line
        m = date_re.search(line)
        if m:
            current_date = m.group(1)
            in_add = False
            in_del = False
        if "ADD" in line.upper() and "DEL" not in line.upper():
            in_add = True
            in_del = False
            continue
        if "DEL" in line.upper() and "ADD" not in line.upper():
            in_del = True
            in_add = False
            continue
        if "TOTAL" in line.upper() or "PERIOD" in line.upper():
            continue
        if current_date is None:
            continue

        if in_add:
            for t in extract_tickers_from_line(line):
                events.append({"effective_date": current_date, "ticker": t, "action": "ADD"})
        elif in_del:
            for t in extract_tickers_from_line(line):
                events.append({"effective_date": current_date, "ticker": t, "action": "DEL"})

    df = pd.DataFrame(events)
    if df.empty:
        return df
    df["effective_date"] = pd.to_datetime(df["effective_date"])
    df = df.sort_values("effective_date").reset_index(drop=True)
    return df

In [19]:
# Find ndx_changes file
DATA_DIR = os.path.join(BASE, "Data")
NDX_LOG = os.path.join(DATA_DIR, "ndx_changes")
if not os.path.isfile(NDX_LOG):
    NDX_LOG = os.path.join(BASE, "ndx_changes")

print(f"Parsing ndx_changes: {NDX_LOG}")
events = parse_ndx_log(NDX_LOG)

# Save events
if _has_parquet:
    events.to_parquet(os.path.join(RAW_DIR, "ndx_events.parquet"), index=False)
events.to_csv(os.path.join(RAW_DIR, "ndx_events.csv"), index=False)

print(f"Parsed {len(events)} events")
print(f"Date range: {events['effective_date'].min().date()} to {events['effective_date'].max().date()}")
print(f"\nFirst 20 events:")
print(events.head(20))

Parsing ndx_changes: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/ndx_changes
Parsed 1303 events
Date range: 2001-03-01 to 2026-02-01

First 20 events:
   effective_date                     ticker action
0      2001-03-01                       BRCD    ADD
1      2001-03-01                       SDLI    DEL
2      2001-03-01  -------------------------    DEL
3      2001-04-01                   1280712D    ADD
4      2001-04-01                        BMC    DEL
5      2001-04-01  -------------------------    DEL
6      2001-06-01                       NVDA    ADD
7      2001-06-01                    166783Q    DEL
8      2001-06-01  -------------------------    DEL
9      2001-11-01                      EXDSQ    DEL
10     2001-11-01                      ATHMQ    DEL
11     2001-11-01  -------------------------    DEL
12     2001-11-01                       ADRX    ADD
13     2001-11-01                       GILD    ADD
14     2002-01-01                   354916

## STEP 2: Load yfinance Retrieved Tickers

In [20]:
# Load tickers that were successfully downloaded from yfinance
tickers_retrieved_path = os.path.join(RAW_DIR, "tickers_retrieved.csv")
tickers_not_retrieved_path = os.path.join(RAW_DIR, "tickers_not_retrieved.csv")

df_retrieved = pd.read_csv(tickers_retrieved_path)
df_not_retrieved = pd.read_csv(tickers_not_retrieved_path)

yfinance_tickers = sorted(df_retrieved['ticker'].tolist())
unavailable_tickers = sorted(df_not_retrieved['ticker'].tolist())

print(f"yfinance-available tickers: {len(yfinance_tickers)}")
print(f"Unavailable tickers: {len(unavailable_tickers)}")
print(f"  - Bloomberg IDs: {len(df_not_retrieved[df_not_retrieved['reason'] == 'Bloomberg ID'])}")
print(f"  - Delisted/Unavailable: {len(df_not_retrieved[df_not_retrieved['reason'] == 'Unavailable/Delisted'])}")

yfinance-available tickers: 194
Unavailable tickers: 143
  - Bloomberg IDs: 44
  - Delisted/Unavailable: 99


## STEP 3: Build Full Universe of Tickers

In [21]:
# Full universe = all tickers from events (including Bloomberg IDs and unavailable)
event_tickers = set(events["ticker"].unique())
all_tickers = sorted(event_tickers | set(yfinance_tickers) | set(unavailable_tickers))

print(f"\nFull universe: {len(all_tickers)} tickers")
print(f"  - Event tickers: {len(event_tickers)}")
print(f"  - yfinance-available: {len(yfinance_tickers)}")
print(f"  - Unavailable: {len(unavailable_tickers)}")

# Identify Bloomberg-only IDs
bloomberg_only = [t for t in all_tickers if re.match(r"^\d{7,8}[A-Z]$", t)]
print(f"\nBloomberg IDs in full universe: {len(bloomberg_only)}")
print(f"Sample: {bloomberg_only[:10]}")


Full universe: 339 tickers
  - Event tickers: 339
  - yfinance-available: 194
  - Unavailable: 143

Bloomberg IDs in full universe: 44
Sample: ['0945329D', '0964591D', '1040983D', '1255459D', '1280712D', '1288453D', '1396924D', '1396926D', '1448062D', '1518855D']


## STEP 4: Determine Initial Membership State

Since we don't have a complete snapshot for 2001-03-15, we'll use the first ADD event for each ticker as its starting point. For tickers that appear in events but have no ADD before our start date, we assume they were already members.

In [22]:
# Find first event date
FIRST_EVENT_DATE = events["effective_date"].min()
START = DATA_START_DATE
END = DATA_END_DATE

print(f"Data range: {START.date()} to {END.date()}")
print(f"First event in log: {FIRST_EVENT_DATE.date()}")

# Build initial membership: tickers that appear in events before or at FIRST_EVENT_DATE
# and don't have a DEL before their first ADD
initial_members = set()
for ticker in event_tickers:
    ticker_events = events[events['ticker'] == ticker].sort_values('effective_date')
    first_event = ticker_events.iloc[0]
    
    # If first event is ADD, or if ticker appears without explicit ADD before FIRST_EVENT_DATE,
    # assume it was a member at START
    if first_event['action'] == 'ADD' and first_event['effective_date'] <= FIRST_EVENT_DATE:
        initial_members.add(ticker)
    elif first_event['effective_date'] > FIRST_EVENT_DATE:
        # Ticker added after first event date, not an initial member
        pass
    else:
        # First event is DEL, so it was a member before
        initial_members.add(ticker)

print(f"\nInitial members (at {START.date()}): {len(initial_members)}")
print(f"Sample: {sorted(list(initial_members))[:20]}")

Data range: 2003-01-01 to 2026-03-15
First event in log: 2001-03-01

Initial members (at 2003-01-01): 3
Sample: ['-------------------------', 'BRCD', 'SDLI']


## STEP 5: Build membership_daily_full (ALL tickers)

In [23]:
def build_membership_daily(
    events: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp,
    all_tickers: list,
    initial_members: set,
) -> pd.DataFrame:
    """Build membership_daily matrix (0/1) from events.
    
    Logic:
    1. Initialize all tickers to 0
    2. Set initial_members to 1 for all dates
    3. Apply ADD/DEL events chronologically
    """
    date_index = pd.date_range(start=start, end=end, freq="D")
    membership = pd.DataFrame(0, index=date_index, columns=all_tickers, dtype=np.int8)
    membership.index.name = "date"

    # STEP 1: Set initial members to 1 for all dates
    for t in initial_members:
        if t in membership.columns:
            membership[t] = 1

    # STEP 2: Apply ADD/DEL events (these override the initial state)
    for _, row in events.iterrows():
        d = row["effective_date"]
        if isinstance(d, str):
            d = pd.Timestamp(d)
        t = row["ticker"]
        if t not in membership.columns:
            continue
        if row["action"] == "ADD":
            membership.loc[membership.index >= d, t] = 1
        else:  # DEL
            membership.loc[membership.index >= d, t] = 0

    membership = membership.clip(0, 1).fillna(0).astype(np.int8)
    return membership


# Build full membership matrix
print("Building membership_daily_full...")
membership_daily_full = build_membership_daily(
    events,
    start=START,
    end=END,
    all_tickers=all_tickers,
    initial_members=initial_members,
)

print(f"membership_daily_full shape: {membership_daily_full.shape}")
print(f"Date range: {membership_daily_full.index[0].date()} to {membership_daily_full.index[-1].date()}")

# Show membership count over time
member_counts = membership_daily_full.sum(axis=1)
print(f"\nMembership count statistics:")
print(member_counts.describe())
print(f"\nSample member counts:")
for d in [membership_daily_full.index[0], membership_daily_full.index[len(membership_daily_full)//2], membership_daily_full.index[-1]]:
    print(f"  {d.date()}: {membership_daily_full.loc[d].sum()} members")

Building membership_daily_full...
membership_daily_full shape: (8475, 339)
Date range: 2003-01-01 to 2026-03-15

Membership count statistics:
count    8475.000000
mean      100.186313
std         3.195997
min        95.000000
25%        98.000000
50%        99.000000
75%       103.000000
max       108.000000
dtype: float64

Sample member counts:
  2003-01-01: 95 members
  2014-08-08: 101 members
  2026-03-15: 99 members


In [24]:
for d in membership_daily_full.index:
    print(f"  {d.date()}: {membership_daily_full.loc[d].sum()} members")

  2003-01-01: 95 members
  2003-01-02: 95 members
  2003-01-03: 95 members
  2003-01-04: 95 members
  2003-01-05: 95 members
  2003-01-06: 95 members
  2003-01-07: 95 members
  2003-01-08: 95 members
  2003-01-09: 95 members
  2003-01-10: 95 members
  2003-01-11: 95 members
  2003-01-12: 95 members
  2003-01-13: 95 members
  2003-01-14: 95 members
  2003-01-15: 95 members
  2003-01-16: 95 members
  2003-01-17: 95 members
  2003-01-18: 95 members
  2003-01-19: 95 members
  2003-01-20: 95 members
  2003-01-21: 95 members
  2003-01-22: 95 members
  2003-01-23: 95 members
  2003-01-24: 95 members
  2003-01-25: 95 members
  2003-01-26: 95 members
  2003-01-27: 95 members
  2003-01-28: 95 members
  2003-01-29: 95 members
  2003-01-30: 95 members
  2003-01-31: 95 members
  2003-02-01: 95 members
  2003-02-02: 95 members
  2003-02-03: 95 members
  2003-02-04: 95 members
  2003-02-05: 95 members
  2003-02-06: 95 members
  2003-02-07: 95 members
  2003-02-08: 95 members
  2003-02-09: 95 members


In [25]:
# Save membership_daily_full
if _has_parquet:
    membership_daily_full.to_parquet(os.path.join(RL_DIR, "membership_daily_full.parquet"))
membership_daily_full.to_csv(os.path.join(RL_DIR, "membership_daily_full.csv"))
print(f"\nSaved membership_daily_full.csv: {membership_daily_full.shape}")


Saved membership_daily_full.csv: (8475, 339)


## STEP 6: Build membership_daily (yfinance-only)

In [26]:
# Filter to only yfinance-available tickers
yfinance_cols = [t for t in membership_daily_full.columns if t in yfinance_tickers]
membership_daily = membership_daily_full[yfinance_cols].copy()

print(f"membership_daily shape: {membership_daily.shape}")
print(f"Filtered from {membership_daily_full.shape[1]} to {membership_daily.shape[1]} tickers")

# Show membership count for yfinance-only
member_counts_yf = membership_daily.sum(axis=1)
print(f"\nyfinance membership count statistics:")
print(member_counts_yf.describe())
print(f"\nSample member counts:")
for d in [membership_daily.index[0], membership_daily.index[len(membership_daily)//2], membership_daily.index[-1]]:
    print(f"  {d.date()}: {membership_daily.loc[d].sum()} tradable members")

membership_daily shape: (8475, 194)
Filtered from 339 to 194 tickers

yfinance membership count statistics:
count    8475.000000
mean       73.946549
std        15.608705
min        48.000000
25%        63.000000
50%        68.000000
75%        90.000000
max        98.000000
dtype: float64

Sample member counts:
  2003-01-01: 48 tradable members
  2014-08-08: 68 tradable members
  2026-03-15: 98 tradable members


In [27]:
# Save membership_daily (yfinance-only)
if _has_parquet:
    membership_daily.to_parquet(os.path.join(RL_DIR, "membership_daily.parquet"))
membership_daily.to_csv(os.path.join(RL_DIR, "membership_daily.csv"))
print(f"\nSaved membership_daily.csv: {membership_daily.shape}")


Saved membership_daily.csv: (8475, 194)


## STEP 7: Build close_prices from Individual Ticker Files

In [28]:
import glob

# Find all ticker CSV files (exclude special files)
SPECIAL_FILES = {'close_prices.csv', 'close_prices.parquet', 'ndx_events.csv', 'ndx_events.parquet',
                 'tickers_retrieved.csv', 'tickers_not_retrieved.csv'}
ticker_files = [f for f in glob.glob(os.path.join(RAW_DIR, "*.csv"))
                if os.path.basename(f) not in SPECIAL_FILES]

print(f"Found {len(ticker_files)} ticker CSV files in {RAW_DIR}")

# Build close_prices by reading each ticker file
close_data = {}
missing_tickers = []
for fpath in ticker_files:
    ticker = os.path.basename(fpath).replace('.csv', '')
    try:
        df = pd.read_csv(fpath)
        if 'close' in df.columns and 'date' in df.columns:
            # Parse date
            df['date'] = pd.to_datetime(df['date'])
            # Set date as index and extract close prices
            series = df.set_index('date')['close']
            close_data[ticker] = series
        else:
            missing_tickers.append(ticker)
    except Exception as e:
        print(f"  Error reading {ticker}: {e}")
        missing_tickers.append(ticker)

print(f"Successfully loaded close prices for {len(close_data)} tickers")
if missing_tickers:
    print(f"Missing/failed tickers: {missing_tickers[:10]}{'...' if len(missing_tickers) > 10 else ''}")

# Combine into a single DataFrame
close_prices = pd.DataFrame(close_data)
close_prices = close_prices.sort_index()
print(f"\nclose_prices shape: {close_prices.shape}")
print(f"close_prices date range: {close_prices.index.min().date()} to {close_prices.index.max().date()}")

Found 194 ticker CSV files in /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/Raw_Data
Successfully loaded close prices for 194 tickers

close_prices shape: (6286, 194)
close_prices date range: 2001-03-15 to 2026-03-13


In [31]:
# Save close_prices
CLOSE_PATH = os.path.join(RL_DIR, "close_prices.parquet")
CLOSE_CSV = os.path.join(RL_DIR, "close_prices.csv")
if _has_parquet:
    close_prices.to_parquet(CLOSE_PATH)
close_prices.to_csv(CLOSE_CSV)
print(f"Saved close_prices to {CLOSE_CSV}")

Saved close_prices to /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/RL_Needs/close_prices.csv


## STEP 8: Build tradable_mask_full and tradable_mask

Tradable = (in NDX) AND (has price data)

In [32]:
# Align membership_daily_full with close_prices dates
# Reindex membership to match close_prices date range
membership_aligned = membership_daily_full.reindex(close_prices.index).fillna(0).astype(np.int8)

# Align close_prices to have all tickers from membership
all_cols = sorted(set(membership_aligned.columns) | set(close_prices.columns))
mh_full = membership_aligned.reindex(columns=all_cols).fillna(0).clip(0, 1).astype(np.int8)
cp_full = close_prices.reindex(columns=all_cols)

print(f"Aligned shapes:")
print(f"  membership: {mh_full.shape}")
print(f"  close_prices: {cp_full.shape}")
print(f"  Date range: {mh_full.index[0].date()} to {mh_full.index[-1].date()}")

Aligned shapes:
  membership: (6286, 339)
  close_prices: (6286, 339)
  Date range: 2001-03-15 to 2026-03-13


In [33]:
# tradable_mask_full = (member) AND (has price)
tradable_mask_full = ((mh_full == 1) & cp_full.notna()).astype(np.int8)

if _has_parquet:
    tradable_mask_full.to_parquet(os.path.join(RL_DIR, "tradable_mask_full.parquet"))
tradable_mask_full.to_csv(os.path.join(RL_DIR, "tradable_mask_full.csv"))

print(f"\ntradable_mask_full shape: {tradable_mask_full.shape}")
print(f"Saved tradable_mask_full.csv")

# Show tradable counts
print(f"\nTradable count at sample dates:")
for ts in [tradable_mask_full.index[0], tradable_mask_full.index[len(tradable_mask_full)//2], tradable_mask_full.index[-1]]:
    mem_count = int(mh_full.loc[ts].sum())
    trad_count = int(tradable_mask_full.loc[ts].sum())
    print(f"  {ts.date()}: {trad_count} tradable / {mem_count} members")


tradable_mask_full shape: (6286, 339)
Saved tradable_mask_full.csv

Tradable count at sample dates:
  2001-03-15: 0 tradable / 0 members
  2013-09-13: 64 tradable / 98 members
  2026-03-13: 98 tradable / 99 members


In [34]:
# tradable_mask (yfinance-only)
yfinance_cols_available = [t for t in yfinance_tickers if t in tradable_mask_full.columns]
tradable_mask = tradable_mask_full[yfinance_cols_available].copy()

if _has_parquet:
    tradable_mask.to_parquet(os.path.join(RL_DIR, "tradable_mask.parquet"))
tradable_mask.to_csv(os.path.join(RL_DIR, "tradable_mask.csv"))

print(f"\ntradable_mask shape: {tradable_mask.shape}")
print(f"Saved tradable_mask.csv (yfinance-only, for RL)")

# Show tradable counts for yfinance-only
print(f"\nTradable count (yfinance-only) at sample dates:")
for ts in [tradable_mask.index[0], tradable_mask.index[len(tradable_mask)//2], tradable_mask.index[-1]]:
    trad_count = int(tradable_mask.loc[ts].sum())
    print(f"  {ts.date()}: {trad_count} tradable")


tradable_mask shape: (6286, 194)
Saved tradable_mask.csv (yfinance-only, for RL)

Tradable count (yfinance-only) at sample dates:
  2001-03-15: 0 tradable
  2013-09-13: 64 tradable
  2026-03-13: 98 tradable


## Sanity Checks

In [35]:
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# Check 1: No tradable where close is NaN
bad = (tradable_mask_full == 1) & cp_full.isna()
assert bad.sum().sum() == 0, "tradable_mask_full has 1 where close is NaN"
print("✓ No tradable_mask=1 where close is NaN")

# Check 2: membership value range
assert membership_daily_full.min().min() == 0
assert membership_daily_full.max().max() == 1
print("✓ membership_daily_full values are 0 or 1")

# Check 3: tradable_mask value range
assert tradable_mask_full.min().min() == 0
assert tradable_mask_full.max().max() == 1
print("✓ tradable_mask_full values are 0 or 1")

# Check 4: Date ranges match
assert membership_daily_full.index[0].date() == START.date()
assert membership_daily_full.index[-1].date() == END.date()
print(f"✓ Date range is {START.date()} to {END.date()}")

# Check 5: Member count should be around 100-102
member_counts = membership_daily_full.sum(axis=1)
avg_members = member_counts.mean()
print(f"✓ Average members per day: {avg_members:.1f} (expected ~100-102)")

print("\n" + "=" * 60)
print("ALL SANITY CHECKS PASSED")
print("=" * 60)

SANITY CHECKS
✓ No tradable_mask=1 where close is NaN
✓ membership_daily_full values are 0 or 1
✓ tradable_mask_full values are 0 or 1
✓ Date range is 2003-01-01 to 2026-03-15
✓ Average members per day: 100.2 (expected ~100-102)

ALL SANITY CHECKS PASSED


## Summary

In [36]:
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nData range: {START.date()} to {END.date()}")
print(f"Total days: {len(membership_daily_full)}")
print(f"\nTickers:")
print(f"  - Full universe: {len(all_tickers)} tickers")
print(f"  - yfinance-available: {len(yfinance_tickers)} tickers")
print(f"  - Unavailable: {len(unavailable_tickers)} tickers")
print(f"\nFiles created:")
print(f"  - membership_daily_full.csv: {membership_daily_full.shape}")
print(f"  - membership_daily.csv: {membership_daily.shape}")
print(f"  - close_prices.csv: {close_prices.shape}")
print(f"  - tradable_mask_full.csv: {tradable_mask_full.shape}")
print(f"  - tradable_mask.csv: {tradable_mask.shape}")
print(f"\nAll files saved to:")
print(f"  - Raw data: {RAW_DIR}")
print(f"  - RL matrices: {RL_DIR}")
print("\n" + "=" * 60)
print("COMPLETE!")
print("=" * 60)


SUMMARY

Data range: 2003-01-01 to 2026-03-15
Total days: 8475

Tickers:
  - Full universe: 339 tickers
  - yfinance-available: 194 tickers
  - Unavailable: 143 tickers

Files created:
  - membership_daily_full.csv: (8475, 339)
  - membership_daily.csv: (8475, 194)
  - close_prices.csv: (6286, 194)
  - tradable_mask_full.csv: (6286, 339)
  - tradable_mask.csv: (6286, 194)

All files saved to:
  - Raw data: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/Raw_Data
  - RL matrices: /Users/kamilkashif/Documents/University/Masters Thesis/Data_Fetch/Data/Outputs/RL_Needs

COMPLETE!


In [37]:
import yfinance as yf
import pandas as pd
import os

# Setup path
RL_DIR = os.path.join(BASE, "Data", "Outputs", "RL_Needs")
os.makedirs(RL_DIR, exist_ok=True)

# Download QQQ
qqq = yf.download('QQQ', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not qqq.empty:
    # Flatten MultiIndex columns if present
    if isinstance(qqq.columns, pd.MultiIndex):
        qqq.columns = qqq.columns.get_level_values(0)
    qqq = qqq.reset_index()
    qqq.columns = [c.lower() for c in qqq.columns]
    qqq.to_csv(os.path.join(RL_DIR, 'QQQ.csv'), index=False)
    print(f"Saved QQQ.csv: {len(qqq)} rows")

Saved QQQ.csv: 5836 rows


In [38]:
# Note: ^FVX is 5-year Treasury yield (closest available on Yahoo Finance)
risk_free = yf.download('^FVX', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not risk_free.empty:
    if isinstance(risk_free.columns, pd.MultiIndex):
        risk_free.columns = risk_free.columns.get_level_values(0)
    risk_free = risk_free.reset_index()
    risk_free.columns = [c.lower() for c in risk_free.columns]
    risk_free.to_csv(os.path.join(RL_DIR, 'risk_free_data.csv'), index=False)
    print(f"Saved risk_free_data.csv: {len(risk_free)} rows")

Saved risk_free_data.csv: 5830 rows


In [39]:
# Download VIX
vix = yf.download('^VIX', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not vix.empty:
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = vix.columns.get_level_values(0)
    vix = vix.reset_index()
    vix.columns = [c.lower() for c in vix.columns]
    vix.to_csv(os.path.join(RL_DIR, 'VIX.csv'), index=False)
    print(f"Saved VIX.csv: {len(vix)} rows")

Saved VIX.csv: 5836 rows


In [41]:
treasury = yf.download('^IRX', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not treasury.empty:
    if isinstance(treasury.columns, pd.MultiIndex):
        treasury.columns = treasury.columns.get_level_values(0)
    treasury = treasury.reset_index()
    treasury.columns = [c.lower() for c in treasury.columns]
    treasury.to_csv(os.path.join(RL_DIR, 'risk_free_data.csv'), index=False)
    print(f"✓ Saved risk_free_data.csv: {len(treasury)} rows from {treasury['date'].min()} to {treasury['date'].max()}")
else:
    print("✗ Failed to download Treasury data")

✓ Saved risk_free_data.csv: 5830 rows from 2003-01-02 00:00:00 to 2026-03-13 00:00:00
